In [ ]:
! pip install av

In [ ]:
import sys
print(sys.version)
import torch
print(torch.__version__)
import transformers as tf
print(tf.__version__)

In [ ]:
import wandb
import av
import numpy as np
import json
import os
import re
import torch
from tqdm import tqdm 
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


from transformers import AutoImageProcessor
from transformers import MBartForConditionalGeneration, TimesformerModel
from transformers import MBart50TokenizerFast
from transformers.modeling_outputs import BaseModelOutput
from transformers import TrainingArguments, Trainer

In [ ]:
os.environ["WANDB_API_KEY"] = "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"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset_folder = fr"/kaggle/input/datasets/sandipsanjel123/msvd-dataset-nepali/msvd_dataset"
video_folder_path = os.path.join(dataset_folder, "videos_train_val")
train_json_path = os.path.join(dataset_folder, "train.json")
val_json_path = os.path.join(dataset_folder, "val.json")
test_json_path = os.path.join(dataset_folder, "test.json")
cache_dir = fr"/kaggle/working/video_cache"

In [ ]:
def read_video_pyav(video_path, num_frames=8):
    container = av.open(video_path)
    frames = []

    for frame in container.decode(video=0):
        frames.append(frame.to_rgb().to_ndarray())

    container.close()

    # uniform sampling
    idx = np.linspace(0, max(len(frames) - 1, 0), num_frames).astype(np.int64)
    frames = [frames[i] for i in idx]

    return frames  # (T, H, W, C)

In [ ]:
tokenizer =MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50", src_lang = "ne_NP", tgt_lang = "ne_NP")
image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")

In [ ]:
def process_video(video_path, image_processor):
    frames = read_video_pyav(video_path)
    inputs = image_processor(list(frames), return_tensors="pt")
    return inputs["pixel_values"].half()# (1, T, C, H, W)

In [ ]:
def make_video_caption_pairs(json_path):
    pairs = []
    with open(json_path, encoding="utf-8") as file:
        data = json.load(file)

    for item in data:
        video_id = item["video_id"]
        captions = item["caption"]
        
        for caption in captions:
                pairs.append({
                    "video_id": video_id,
                    "caption": caption
                })
    return pairs


In [ ]:
train_pairs = make_video_caption_pairs(train_json_path)
val_pairs = make_video_caption_pairs(val_json_path)

In [ ]:
class VideoOnlyDataset(Dataset):
    def __init__(self, pairs, video_dir, cache_dir, processor):
        seen = set()
        self.items = []
        for p in pairs:
            if p["video_id"] not in seen:
                if not os.path.exists(os.path.join(cache_dir, f"{p['video_id']}.pt")):
                    seen.add(p["video_id"])
                    self.items.append(p["video_id"])

        self.video_dir = video_dir
        self.cache_dir = cache_dir
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        video_id   = self.items[idx]
        video_path = os.path.join(self.video_dir, f"{video_id}.avi")
        tensor     = process_video(video_path, self.processor)  # [1, 8, 3, 224, 224] fp16
        tensor     = tensor.squeeze(0)  # [8, 3, 224, 224] fp16
        return video_id, tensor

In [ ]:
def cache_collate_fn(batch):
    video_ids = [item[0] for item in batch]
    tensors   = torch.stack([item[1] for item in batch])  # [B, 8, 3, 224, 224]
    return video_ids, tensors


def build_video_cache(pairs, video_dir, cache_dir, processor, num_workers=8, batch_size=16):
    os.makedirs(cache_dir, exist_ok=True)
    print(f"Building cache in: {cache_dir}")

    dataset = VideoOnlyDataset(pairs, video_dir, cache_dir, processor)

    if len(dataset) == 0:
        print("All videos already cached ✅")
        return

    loader = DataLoader(
        dataset,
        batch_size  = batch_size,
        num_workers = num_workers,
        shuffle     = False,
        collate_fn  = cache_collate_fn
    )

    print(f"Caching {len(dataset)} videos...")

    for video_ids, tensors in tqdm(loader, total=len(loader)):
        # verify dtype first time
        for video_id, tensor in zip(video_ids, tensors):
            cache_path = os.path.join(cache_dir, f"{video_id}.pt")
            torch.save(tensor.clone(), cache_path, _use_new_zipfile_serialization=False)

    print("Done ✅")

In [ ]:
build_video_cache(
    pairs      = train_pairs,
    video_dir  = video_folder_path,
    cache_dir  = cache_dir,
    processor  = image_processor,
    num_workers= 8,     
    batch_size = 16
)

# Val set
build_video_cache(
    pairs      = val_pairs,
    video_dir  = video_folder_path,
    cache_dir  = cache_dir,
    processor  = image_processor,
    num_workers= 8,    
    batch_size = 16
)


In [ ]:
import psutil

def get_ram_usage():
    ram = psutil.virtual_memory()
    print(f"Used RAM : {ram.used/1e9:.1f} GB")
    print(f"Free RAM : {ram.available/1e9:.1f} GB")

print("=== BEFORE ===")
get_ram_usage()

video_cache = {}
for filename in tqdm(os.listdir(cache_dir)):
    if filename.endswith(".pt"):
        video_id = filename.replace(".pt", "")
        video_cache[video_id] = torch.load(
            os.path.join(cache_dir, filename),
            weights_only=False,
            map_location="cpu"
        )

print("=== AFTER ===")
get_ram_usage()
print(f"Videos loaded: {len(video_cache)}")

In [ ]:
class Dataset(Dataset):
    def __init__(
        self,
        pairs,
        video_cache,
        tokenizer,
        max_length=50
    ):
        self.pairs = pairs
        self.video_cache = video_cache
        self.tokenizer = tokenizer
        self.max_length = max_length


    def __len__(self):
        return len(self.pairs)
        
    def __getitem__(self, idx):
        item = self.pairs[idx]
        video_id = item["video_id"]
        caption = item["caption"]
    
        video_tensor = video_cache[video_id].float()
        
        # Tokenize caption
        text_inputs = self.tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        input_ids = text_inputs["input_ids"].squeeze(0)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
    
        return {
            "pixel_values": video_tensor,
            "input_ids": input_ids,
            "labels": labels
        }

In [ ]:
print(len(tokenizer))

In [ ]:
# Check special tokens
print("BOS:", tokenizer.bos_token)
print("EOS:", tokenizer.eos_token)
print("PAD:", tokenizer.pad_token)

In [ ]:
# Check special tokens
print("BOS_id:", tokenizer.bos_token_id)
print("EOS_id:", tokenizer.eos_token_id)
print("PAD_id:", tokenizer.pad_token_id)

In [ ]:
train_dataset = Dataset(train_pairs, video_cache, tokenizer)
val_dataset   = Dataset(val_pairs, video_cache, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle = True, pin_memory = True, num_workers=4, persistent_workers=True)
val_loader   = DataLoader(val_dataset, batch_size=2, pin_memory = True, num_workers=4, persistent_workers=True)

In [ ]:
for batch in train_loader:
    print(batch["pixel_values"].shape)
    print(batch["labels"].shape)
    print(batch["input_ids"].shape)
    break

In [ ]:
sample = train_dataset[20000]
print("Video ID Caption Pair:")
# print("Video_ID:", sample["Video_ID"])
print("Video Tensor: ", sample["pixel_values"])
print("Video Tensor: ", sample["pixel_values"].shape)
print("Input IDs : ", sample["input_ids"])
print("Input IDs : ", sample["input_ids"].shape)
print("Labels: ", sample["labels"])
print("Labels shape: ", sample["labels"].shape)

In [ ]:
# Ensure tensor is on CPU and float32
frames = sample["pixel_values"].detach().cpu().float()  # [T, C, H, W]

# Denormalize frames (inverse of typical ImageNet normalization)
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
frames = frames * std + mean

# Clamp values to [0,1] for safe visualization
frames = frames.clamp(0, 1)

# Convert to numpy and rearrange dimensions for matplotlib
frames = frames.numpy()  # [T, C, H, W]
frames = np.transpose(frames, (0, 2, 3, 1))  # [T, H, W, C]

# Create subplots
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flat  # Flatten axes for iteration

for idx, ax in enumerate(axes):
    if idx < len(frames):
        ax.imshow(frames[idx])
        ax.set_title(f'Frame {idx+1}')
        ax.axis('off')
    else:
        ax.axis('off')

plt.suptitle('Video Frames (8 frames)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
text = tokenizer.decode(sample["input_ids"])
print("Decoded Text:", text)

In [ ]:
print(len(train_dataset))
print(len(val_dataset))

In [ ]:
for token_id in sample["input_ids"][:20]:  # first 20 tokens
    print(token_id.item(), "→", tokenizer.convert_ids_to_tokens([token_id.item()])[0])

In [ ]:
torch.cuda.empty_cache()  

In [ ]:
# --------------------------------------------------
# Q-Former Block
# --------------------------------------------------
class QFormerBlock(nn.Module):
    def __init__(self, hidden_dim=1024, num_heads=8, mlp_ratio=4.0):
        super().__init__()

        # Cross-attention (Queries attend to video features)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(hidden_dim)

        # Self-attention among queries
        self.self_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

        # Feed-forward
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, int(hidden_dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(hidden_dim * mlp_ratio), hidden_dim),
        )
        self.norm3 = nn.LayerNorm(hidden_dim)

    def forward(self, queries, video_features):
        # Cross-attention
        cross_out, _ = self.cross_attn(
            query=queries,
            key=video_features,
            value=video_features,
        )
        queries = self.norm1(queries + cross_out)

        # Self-attention
        self_out, _ = self.self_attn(
            query=queries,
            key=queries,
            value=queries,
        )
        queries = self.norm2(queries + self_out)

        # MLP
        mlp_out = self.mlp(queries)
        queries = self.norm3(queries + mlp_out)

        return queries


# --------------------------------------------------
# Video Captioning Model with Q-Former Bridge
# --------------------------------------------------
class VideoCaptioningModel(nn.Module):
    def __init__(
        self,
        timesformer_model_name="facebook/timesformer-base-finetuned-k600",
        mbart_model_name="facebook/mbart-large-50",
        num_query_tokens=32,
        qformer_layers=4,
        freeze_timesformer=True,
        freeze_mbart_encoder=True,
    ):
        super().__init__()

        # -------- Encoder: TimeSformer --------
        self.encoder = TimesformerModel.from_pretrained(timesformer_model_name)

        # -------- Decoder: mBART --------
        self.decoder = MBartForConditionalGeneration.from_pretrained(mbart_model_name)

        ts_hidden = self.encoder.config.hidden_size        # usually 768
        mbart_hidden = self.decoder.config.d_model         # 1024

        # -------- Projection (768 → 1024) --------
        self.video_proj = nn.Linear(ts_hidden, mbart_hidden)

        # -------- Q-Former --------
        self.query_tokens = nn.Parameter(
            torch.randn(1, num_query_tokens, mbart_hidden)
        )

        self.qformer_blocks = nn.ModuleList(
            [QFormerBlock(hidden_dim=mbart_hidden) for _ in range(qformer_layers)]
        )

        # -------- Freeze modules --------
        if freeze_timesformer:
            for p in self.encoder.parameters():
                p.requires_grad = False

        if freeze_mbart_encoder:
            for p in self.decoder.model.encoder.parameters():
                p.requires_grad = False
            for p in self.decoder.model.decoder.parameters():
                p.requires_grad = True

    # --------------------------------------------------
    # Forward (Training)
    # --------------------------------------------------
    def forward(self, pixel_values, input_ids, labels):

        # 1️⃣ TimeSformer Encoding
        encoder_outputs = self.encoder(
            pixel_values=pixel_values,
            return_dict=True
        )

        video_features = encoder_outputs.last_hidden_state  # (B, N, 768)

        # 2️⃣ Project to 1024
        video_features = self.video_proj(video_features)

        B = video_features.size(0)

        # 3️⃣ Expand Query Tokens
        queries = self.query_tokens.expand(B, -1, -1)

        # 4️⃣ Q-Former Processing
        for block in self.qformer_blocks:
            queries = block(queries, video_features)

        # queries shape: (B, num_query_tokens, 1024)

        # 5️⃣ Wrap as Encoder Memory for mBART
        encoder_outputs = BaseModelOutput(
            last_hidden_state=queries
        )
        
        encoder_attention_mask = torch.ones(
        queries.size()[:-1],
        dtype=torch.long,
        device=queries.device
        )

        # 6️⃣ mBART Decoder
        outputs = self.decoder(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            input_ids=input_ids,
            labels=labels,
            return_dict=True
        )

        return outputs

    # --------------------------------------------------
    # Generate (Inference)
    # --------------------------------------------------
    @torch.no_grad()
    def generate(
        self,
        pixel_values,
        max_length=50,
        num_beams=4,
        forced_bos_token_id=None,
        decoder_start_token_id=None,
        **generate_kwargs
    ):

        encoder_outputs = self.encoder(
            pixel_values=pixel_values,
            return_dict=True
        )

        video_features = self.video_proj(
            encoder_outputs.last_hidden_state
        )

        B = video_features.size(0)
        queries = self.query_tokens.expand(B, -1, -1)

        for block in self.qformer_blocks:
            queries = block(queries, video_features)

        encoder_outputs = BaseModelOutput(
            last_hidden_state=queries
        )

        encoder_attention_mask = torch.ones(
        queries.size()[:-1],
        dtype=torch.long,
        device=queries.device
       ) 

        generated_ids = self.decoder.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_length,
            num_beams=num_beams,
            forced_bos_token_id=forced_bos_token_id,
            decoder_start_token_id=decoder_start_token_id,
            **generate_kwargs
        )

        return generated_ids

In [ ]:
# ==================== MODEL ====================
model = VideoCaptioningModel()


In [ ]:
from torchinfo import summary
summary(model, input_size = [(1,8,3,224,224),(1,50),(1,50)] , dtypes=[torch.float32,torch.long,torch.long])

In [ ]:
def data_collator(batch):
    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),  
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }


In [ ]:
# ckpt_dir = "/kaggle/input/datasets/sandipsanjel123/checkpoint-qformer-6000-70-15-15/checkpoints/checkpoint-6000"

In [ ]:
# # For resuming with the curve
# wandb.init(
#     project="huggingface",
#     id="c5gupjli",
#     resume="allow"
# )

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    #Training
    num_train_epochs = 5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,
    gradient_accumulation_steps=2,

    #Optimization
    learning_rate = 1e-5,
    weight_decay = 0.001,
    fp16 = True,

    #Evaluation

    eval_strategy = "steps",
    eval_steps = 1000,
    save_strategy = "steps",
    save_steps = 1000,
    logging_strategy = "steps",
    logging_steps = 1000,

    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    save_total_limit = 2,
    save_safetensors = False,
    report_to = "wandb",
    run_name = "timesformer-qformer-mbart-vatex-a100",
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,         
    data_collator=data_collator,
    processing_class=tokenizer,
)


In [ ]:
# trainer.train(resume_from_checkpoint=ckpt_dir)

In [ ]:
trainer.train()